# 大模型调用器

上一章，小明的学习助手通过 OpenAI 的 API 和大模型完成了第一次讲话。

事实上，大模型不止一家：OpenAI 的 GPT、Anthropic 的 Claude、Google 的 Gemini，还有 DeepSeek、Kimi、Qwen、MiniMax 这些国产开源模型……

小明想：要是学习助手能随时换用不同的模型就好了。比如平时用便宜的小模型，遇到难题再换一个更强的。

问题是，每家厂商的 API 长得都不太一样。

这一章，我们先看一看 Claude 的 API 有什么不同，然后给不同厂商的 API 封装进**统一的调用器**：不管用哪家的模型，调用方式都一模一样。

## Claude 准备工作

调用 Claude 和调用 OpenAI 一样，需要一个开发者账号和 API Key，步骤也差不多：

1. 打开 Claude 的开发者平台 [platform.claude.com](https://platform.claude.com)，注册或登录。注意，这个开发者账号和聊天用的 claude.ai 是分开的；
2. 绑定付款方式，充值一小笔钱（比如：$5）。和 OpenAI 一样，API 按 token 收费；
3. 进入 **Settings → API keys**，点击 **Create key**，复制生成的 Key。Claude 的 Key 以 `sk-ant-` 开头，同样只显示一次。

然后像上一章一样，把 Key 保存到环境变量里，名字叫 `ANTHROPIC_API_KEY`。

**macOS / Linux：**

```bash
echo 'export ANTHROPIC_API_KEY="sk-ant-..."' >> ~/.zshrc
source ~/.zshrc
```

**Windows（PowerShell）：**

```powershell
setx ANTHROPIC_API_KEY "sk-ant-..."
```

最后安装 Anthropic 官方的 Python 库：

```bash
pip install anthropic
```

记得重新启动 Jupyter，然后检查一下两个 Key 是否都设置好了：

In [1]:
import os

print("OPENAI_API_KEY" in os.environ, "ANTHROPIC_API_KEY" in os.environ)

True True


开始调用 Claude API 之前，需要先导入 OpenAI 和 Anthropic 的库。

In [2]:
from abc import ABC, abstractmethod

import anthropic
from anthropic.types import UserMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionUserMessageParam

## 大模型调用器

两家的 API 大体相同：都要指定模型 `model`，都把对话放在 `messages` 列表里，每条消息都有 `role` 和 `content`。区别在于：

|              | OpenAI | Claude |
|--------------|---|---|
| 调用函数         | `client.chat.completions.create(...)` | `client.messages.create(...)` |
| 最多生成 token 数 | 可以不写 | **必须写** `max_tokens` |
| 输出信息         | `response.choices[0].message.content` | `response.content[0].text` |

后面我们还会遇到更多的差别。

### 抽象类

所以我们将定义一个**抽象类**来定义统一调用接口。`LLMProvider` 继承了 `ABC`（Abstract Base Class，抽象基类），目前只定义一个抽象接口：

* `call`（API 调用函数）：调用大模型 API。

In [3]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, message, model=None):
        pass

### OpenAI 调用器

我们将 OpenAI API 客户端改造成 `OpenAIProvider`：

- `__init__`：创建客户端，我们设置默认模型为 `gpt-5.4-mini`；
- `call`：调用 OpenAI API。可以指定本次调用的模型，或者使用缺省模型。输入信息是用户的提问，输出信息是模型的回答。

In [4]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, message, model=None):
        response = self.client.chat.completions.create(
            model=model or self.default_model,
            messages=[ChatCompletionUserMessageParam(role="user", content=message)]
        )
        return response.choices[0].message.content

用起来非常简单，一行代码就能拿到回答：

In [5]:
provider = OpenAIProvider()
print(provider.call("你好，我叫小明，正在自学编程。请用两三句话介绍一下你自己。"))

你好，小明！我是一个人工智能助手，可以帮你解答编程、学习和写作等问题，也能一起梳理思路、调试代码。你在自学编程的路上如果遇到任何难题，都可以直接问我。


### Claude 调用器

`ClaudeProvider` 的结构和 `OpenAIProvider` 完全一样，只是在 `call` 内部换成了 Claude 的调用方式：

- 用 `anthropic.Anthropic` 创建客户端，Key 来自环境变量 `ANTHROPIC_API_KEY`。
- 默认模型是 `claude-haiku-4-5`，这是 Claude 家族里又快又便宜的模型。
- 调用时必须给出 `max_tokens`，这里默认最多生成 4096 个 token。
- 回答从 `response.content[0].text` 取出。

对外，它和 `OpenAIProvider` 一样，都只有一个 `call` 方法。

In [6]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, message, model=None, max_tokens=4096):
        response = self.client.messages.create(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=[UserMessageParam(role="user", content=message)]
        )
        return response.content[0].text

试一试，同样的问题，Claude 会给出怎样不同的回答：

In [7]:
provider = ClaudeProvider()
print(provider.call("你好，我叫小明，正在自学编程。请用两三句话介绍一下你自己。"))

你好小明！很高兴认识你。我是Claude，一个由Anthropic公司开发的AI助手，能够帮助你解答问题、讨论想法、进行代码编写和调试等多方面的工作。在你自学编程的过程中，我很乐意为你答疑解惑、提供代码建议和学习指导。


## 大模型调用

现在，两个模型的用法完全一样了。不管 `provider` 是哪一个，调用的代码都不用改：

In [8]:
for provider in [OpenAIProvider(), ClaudeProvider()]:
    print(type(provider).__name__)
    print(provider.call("用一句话告诉我，学编程最重要的是什么？"))
    print()

OpenAIProvider
学编程最重要的是：**持续动手练习，用“试错 + 复盘”把问题拆开并解决。**

ClaudeProvider
学编程最重要的是**动手写代码**——通过大量实践解决真实问题，而不是只看书或听课。



## 其他主要的大模型

除了 GPT 和 Claude，市面上还有很多大模型。大致可以分成两类：

**闭源模型**：模型本身不公开，只能通过厂商的 API 使用。

| 模型 | 厂商 | 说明 |
|---|---|---|
| GPT | OpenAI（美国） | 最早流行起来的大模型，ChatGPT 背后的模型 |
| Claude | Anthropic（美国） | 擅长写代码、长文本和智能体任务 |
| Gemini | Google（美国） | 和 Google 的搜索、文档等产品结合紧密 |

**开放权重模型**：模型的参数公开发布，可以下载下来，在自己的电脑或服务器上运行。很多厂商同时也提供 API。

| 模型 | 厂商 | 说明 |
|---|---|---|
| Llama | Meta（美国） | 影响很大的开放模型系列 |
| Mistral | Mistral AI（法国） | 欧洲的代表性模型 |
| DeepSeek | 深度求索（中国） | 以低成本、高性能著称 |
| Qwen（通义千问） | 阿里巴巴（中国） | 型号齐全，从小模型到大模型都有 |

国内还有 Kimi（月之暗面）、GLM（智谱）、豆包（字节跳动）等。大模型更新得很快，各家的具体型号几个月就会变一次，用的时候以各家官网为准。

面对这么多不同的大模型，好消息是：

- **很多厂商的 API 都兼容 OpenAI 的格式。** 用 `OpenAI` 客户端，把 `base_url` 换成它们的地址、`api_key` 换成它们的 Key、`model` 换成它们的模型名，就能直接调用。也就是说，`OpenAIProvider` 稍作修改，就能用上这些模型。
- **开放权重模型可以在本地运行。** 比如用 [Ollama](https://ollama.com) 这样的工具，在自己电脑上下载并运行模型，它也提供兼容 OpenAI 格式的接口。不用花钱，数据也不会离开你的电脑，只是本地的小模型在能力上通常比不过大厂的旗舰模型。

## 动手试试

- 给 `call` 传入 `model` 参数，换一个模型试试，比较一下回答有什么不同。
- 如果你有其他兼容 OpenAI 格式的模型（比如国产模型的 API，或者本地的 Ollama），模仿 `OpenAIProvider` 写一个新的子类：只要改 `base_url`、`api_key` 和默认模型名。

## 小结

- 不同厂商的 API 大同小异：都有 `model` 和 `messages`，但调用方法、必填参数和回答的位置各不相同；
- 我们用**抽象类** `LLMProvider` 规定统一的接口 `call`；
- 学习助手的其他代码只依赖 `LLMProvider`，换模型只需要换一个 `provider`；
- 大模型分为闭源模型和开放权重模型；很多模型兼容 OpenAI 的 API 格式，也可以在本地运行。

现在的 `call` 只能传入一句用户提问，可真正的对话内容要丰富得多。下一章，我们将通过**消息**类，让大模型可以区别不同的信息。